### **Setup**

In [7]:
import os, warnings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import tool 
warnings.filterwarnings("ignore", message="create_react_agent has been moved")
from langgraph.prebuilt import create_react_agent

In [8]:
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPEN_AI_API")

### **Build RAG Chain**

In [10]:
# Init the llm and embeddings model
llm = ChatOpenAI(model = "gpt-4o-mini" , temperature=0 , api_key = api_key)
embeddings = OpenAIEmbeddings(api_key = api_key)

# Load the documents
loader = PyPDFLoader("DIABETES.pdf")
docs = loader.load()

# Split the documents
text_Splitter = RecursiveCharacterTextSplitter(chunk_size = 500 , chunk_overlap = 50)
texts = text_Splitter.split_documents(docs)

# Create a vector store using FAISS
vectorstore = FAISS.from_documents(texts , embedding=embeddings)
retriever = vectorstore.as_retriever()

# 4. Create the RAG Chain using LCEL (LangChain Expression Language)
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

# For formatting the retrived chunks convert them into one string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain creation 
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

### **Define Tools and Agent**

In [11]:
@tool
def rag_gpt_tool(question: str) -> str:
    """Use this tool to answer questions about the 'diabetes' paper. 
    It can answer any question related to this topic from that original paper."""
    return rag_chain.invoke(question)

tools = [rag_gpt_tool]

# 6. Create the ReAct Agent using LangGraph
agent = create_react_agent(llm, tools)

### **Running the Agent**

In [12]:
question = "What is total cost of hospitalization for people with diabetes and diabetic complications in Louisiana? short answer please just give me figure in number nothing more."

print("=" * 60)
print(f"Question: {question}")
print("=" * 60)

# Stream the agent's response to see reasoning steps
for chunk in agent.stream({"messages": [("user", question)]}):
    if "agent" in chunk:
        msg = chunk["agent"]["messages"][0]
        # Check if agent is calling a tool
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"\n🔧 Action: {tool_call['name']}")
                print(f"   Input: {tool_call['args']}")
        # Print agent's reasoning/response
        if msg.content:
            print(f"\n💭 Agent: {msg.content}")
    elif "tools" in chunk:
        tool_msg = chunk["tools"]["messages"][0]
        print(f"\n📋 Observation: {tool_msg.content}")

print("\n" + "=" * 60)
print("Done!")
print("=" * 60)

Question: What is total cost of hospitalization for people with diabetes and diabetic complications in Louisiana? short answer please just give me figure in number nothing more.

🔧 Action: rag_gpt_tool
   Input: {'question': 'What is the total cost of hospitalization for people with diabetes and diabetic complications in Louisiana?'}

📋 Observation: The total cost of hospitalization for people with diabetes and diabetic complications in Louisiana in 2010 was approximately $231,000,000.

💭 Agent: 231000000

Done!


In [13]:
question = "What is diabetes?"

print("=" * 60)
print(f"Question: {question}")
print("=" * 60)

# Stream the agent's response to see reasoning steps
for chunk in agent.stream({"messages": [("user", question)]}):
    if "agent" in chunk:
        msg = chunk["agent"]["messages"][0]
        # Check if agent is calling a tool
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"\n🔧 Action: {tool_call['name']}")
                print(f"   Input: {tool_call['args']}")
        # Print agent's reasoning/response
        if msg.content:
            print(f"\n💭 Agent: {msg.content}")
    elif "tools" in chunk:
        tool_msg = chunk["tools"]["messages"][0]
        print(f"\n📋 Observation: {tool_msg.content}")

print("\n" + "=" * 60)
print("Done!")
print("=" * 60)

Question: What is diabetes?

🔧 Action: rag_gpt_tool
   Input: {'question': 'What is diabetes?'}

📋 Observation: Diabetes is a metabolic disorder characterized by high levels of sugar in the blood, a condition known as hyperglycemia. It occurs when the body is unable to properly regulate blood glucose levels, often due to issues with insulin production or function.

💭 Agent: Diabetes is a metabolic disorder characterized by high levels of sugar in the blood, known as hyperglycemia. It occurs when the body is unable to properly regulate blood glucose levels, often due to issues with insulin production or function.

Done!
